In [ ]:
import cmocean.cm as cmo
import numpy as np
import sys
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from geopy.distance import geodesic
import os
import pandas as pd
import cartopy.crs as ccrs
from pathlib import Path
from matplotlib_scalebar.scalebar import ScaleBar
current_dir = Path.cwd()
src_path = Path(current_dir).parent / "src"
sys.path.insert(0, str(src_path))

import fetch_satellite_singlepass

Load new satellite data

In [ ]:
fetch_satellite_singlepass.fetch_metop_sst_attachments()

## Some plotting functions

In [ ]:
def plot_geo_features(fig, ax, *, zoomed_dim_plot, scalebar_km: float = 100,
                           scalbar_location: str = "lower left") -> None:
    # Formatters for lat/lon labels
    ax.set_extent(zoomed_dim_plot, crs=ccrs.PlateCarree())

    gl = ax.gridlines(color='tab:gray', alpha=0.5, linestyle='--', draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.xlabels_top = False
    gl.ylabels_right = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    
    land = cfeature.LAND.with_scale('50m') #10m also works, but is slower
    ax.add_feature(land, facecolor='#dce2dfff', zorder = 2) #the data is not prossed at all, so there might be sst values on land
    
    add_scalebar(ax, scalebar_km=scalebar_km, location=scalbar_location)
    
    
def add_scalebar(ax, *, scalebar_km: float, location: str = "lower left") -> None:

    # Projected axes are typically in meters; one data unit ≈ 1 m.
    # Convert to km for display: 1 m = 0.001 km
    sb = ScaleBar(
        dx=0.001,               # km per data unit (meter)
        units="km",             # display in kilometers
        fixed_value=scalebar_km,
        location=location,
        box_alpha=0.3,
    )

    ax.add_artist(sb)

In [ ]:
def plot_Sat(ds, zoomed_dim_plot, plot_projection, v_lim = [15, None], title=None):
    
    # Voto planned track
    lats = [57.9313, 58.0308]
    lons = [9.6077, 9.3833]

    data_var_name = "mcsst"   
    fig = plt.figure(figsize=(10, 8))

    ax = plt.axes(projection=plot_projection)

    im = ax.pcolormesh(ds['lon'], ds['lat'], ds[data_var_name][0,:,:],cmap = cmo.thermal, transform=ccrs.PlateCarree(), vmin=v_lim[0], vmax=v_lim[1], zorder = 1)
    cbar = plt.colorbar(im)
    cbar.set_label(data_var_name+ '\°C') 
    

    if title:
        plt.title(title)

    plot_geo_features(fig, ax, zoomed_dim_plot = zoomed_dim_plot)
    ax.scatter(lons, lats, color="red", zorder=10, transform=ccrs.PlateCarree(), label = "planned VOTO devices track")


    out_dir = Path("figures_sat_singlepass")

    if title:
        out_file = out_dir / f"{title}.png"
        plt.savefig(out_file)

    plt.show()
    plt.close(fig)

# Things to change for the user

In [ ]:
zoomed_dim_plot = [7.5, 12, 56.5, 60]  

v_lim = [16,19] # Temperature limits

central_longitude = zoomed_dim_plot[1] - (zoomed_dim_plot[1] - zoomed_dim_plot[0]) / 2
central_latitude = zoomed_dim_plot[2] - (zoomed_dim_plot[2] - zoomed_dim_plot[3]) / 2

plot_projection = ccrs.AzimuthalEquidistant(central_longitude=central_longitude, central_latitude=central_latitude) # Equal distance projection -> needs the center 


### Make relevant pathes 


In [ ]:
(Path("figures_sat_singlepass") / "singlepass").mkdir(parents=True, exist_ok=True)
(Path("figures_sat_singlepass") / "dm").mkdir(parents=True, exist_ok=True)
ground_path = Path.cwd().parent / "data" / "sat_singlepass"

# Plot every Satellite picture that was not plotted in this settings (v_lim, zoomed_dim)

The red scatter is the planned track for the voto devices (but somehow making a label takes 1 minute per figure).

In [ ]:
ground_path = Path("..") / "data" / "sat_singlepass"
out_dir = Path("figures_sat_singlepass")
out_dir.mkdir(parents=True, exist_ok=True)


# loop for singlepass
for file in ground_path.glob("*singlepass.nc"):
    title = f"singlepass/{file.stem[-12:]}-vlim{v_lim}-zoomed_dim{zoomed_dim_plot}"
    out_file = out_dir / f"{title}.png"

    if not out_file.exists():
        with xr.open_dataset(file) as ds:
            plot_Sat(ds, zoomed_dim_plot, plot_projection, v_lim=v_lim, title=title)

# loop for _dm.nc
for file in ground_path.glob("*_dm.nc"):
    title = f"dm/{file.stem[-12:]}-vlim{v_lim}-zoomed_dim{zoomed_dim_plot}"
    out_file = out_dir / f"{title}.png"

    if not out_file.exists():
        with xr.open_dataset(file) as ds:
            plot_Sat(ds, zoomed_dim_plot, plot_projection, v_lim=v_lim, title=title)



# Plot only one dataset


In [ ]:
file = r'03_202509052000_singlepass.nc' # change here to the desired file

nc_path = ground_path / file
ds = xr.open_dataset(nc_path)

v_lim_temp = [16.5, 18.5]
zoomed_dim_plot_temp = [8.5, 11, 57.5, 58.5]

title = str(file[-28:-3]) + "-vlim" + str(v_lim_temp)+ "-zoomed_dim" + str(zoomed_dim_plot_temp)

plot_Sat(ds, zoomed_dim_plot_temp, plot_projection=plot_projection, v_lim=v_lim_temp, title=title)